# 📊 Valores Atípicos en Datos Aseguradores
## Diplomado en Machine Learning para Seguros — Módulo 2 · Tema 1

---

### Contenido del Notebook

| Sección | Tema | Código |
|---------|------|--------|
| **Tema 1** | Métodos Estadísticos de Detección | 1.1 Z-Score · 1.2 Z-Score Modificado (MAD) · 1.3 IQR · 1.4 Percentiles |
| **Tema 2** | Visualización de Outliers | 2.1 Boxplot · 2.2 Scatterplot · 2.3 Q-Q Plot · 2.4 Densidad |

---

**Objetivo:** Que el alumno domine la detección, visualización y manejo de valores atípicos en datos de seguros, comprendiendo cuándo un outlier es un error, cuándo es fraude, y cuándo es un riesgo catastrófico legítimo que debe modelarse.

**⚠️ Recuerda:** En seguros, no todos los outliers son errores. Algunos representan riesgos catastróficos reales que deben modelarse con cuidado.

---
# 🔢 TEMA 1: Métodos Estadísticos para Detección de Outliers

Los métodos estadísticos nos permiten **cuantificar** qué tan atípico es un valor. Cada método tiene supuestos diferentes y detecta distintos tipos de outliers. Vamos a comparar cuatro métodos fundamentales:

- **Z-Score:** Basado en media y desviación estándar (asume normalidad)
- **Z-Score Modificado (MAD):** Robusto, basado en mediana (no asume normalidad)
- **IQR (Rango Intercuartil):** Método del boxplot, basado en cuartiles
- **Percentiles:** El asegurador define el umbral (P95, P99)

## 1.1 Método Z-Score

El Z-Score mide cuántas desviaciones estándar se aleja un valor de la media:

$$Z = \frac{X - \mu}{\sigma}$$

**Regla general:** Si |Z| > 3, el valor se considera un outlier (está a más de 3 desviaciones estándar de la media).

**Limitaciones:**
- Asume distribución normal
- Es **sensible a los propios outliers** (la media y σ se inflan por valores extremos — problema circular)
- En datos de seguros con distribuciones de cola pesada, puede fallar en detectar outliers reales

In [ ]:
import numpy as np
from scipy import stats


# Datos: montos de siniestros de autos (en miles)
siniestros = np.array([10, 12, 11, 13, 50])

# Calcular Z-scores
z_scores = np.abs(stats.zscore(siniestros))

# Identificar outliers (|Z| > 1.5)
outliers = np.where(z_scores > 1.5)[0]

print(f"Z-scores: {z_scores}")
print(f"Índices de outliers: {outliers}")
print(f"Siniestros atípicos: {siniestros[outliers]}")

### 💡 Observa:
El siniestro de $50K tiene un Z-Score de ~2.0, que **no supera el umbral de 3**. Esto sucede porque con solo 5 datos, la desviación estándar se infla por el propio outlier. El Z-Score clásico tiene este problema circular.

**Pregunta para reflexión:** Si cambiaras el umbral a |Z| > 2, ¿detectarías el siniestro de 50K? ¿Es razonable bajar el umbral?

---
## 1.2 Z-Score Modificado (MAD)

El Z-Score Modificado usa la **mediana** y la **Desviación Absoluta Mediana (MAD)** en lugar de la media y σ:

$$M_i = \frac{0.6745 \times (X_i - \text{Mediana})}{\text{MAD}}$$

Donde MAD = Mediana(|Xi − Mediana(X)|)

**Umbral:** |M| > 3.5 indica un outlier.

**Ventajas sobre Z-Score:**
- **Robusto:** La mediana no se distorsiona por outliers extremos
- **No asume normalidad:** Funciona bien con distribuciones sesgadas (típicas en seguros)
- El factor 0.6745 hace que MAD sea consistente con σ bajo normalidad

In [ ]:
import numpy as np
from scipy.stats import median_abs_deviation

# Datos
siniestros = np.array([5, 8, 10, 12, 15, 18, 22, 25, 28, 100])

# Calcular mediana
mediana = np.median(siniestros)

# Calcular MAD
mad = median_abs_deviation(siniestros)

# Calcular M-scores modificados
m_scores = 0.6745 * np.abs(siniestros - mediana) / mad

# Identificar outliers (M > 3.5)
outliers = np.where(m_scores > 3.5)[0]

print(f"Mediana: {mediana}")
print(f"MAD: {mad}")
print(f"M-scores: {m_scores}")
print(f"Outliers detectados: {siniestros[outliers]}")

### 💡 Observa:
El valor de 100K ahora sí es detectado como outlier (M-score ≈ 11.26, muy por encima de 3.5). La mediana (16.5) no fue distorsionada por el valor extremo de 100K, a diferencia de la media que sí lo sería.

**Conclusión:** Para datos de seguros con distribuciones sesgadas, **MAD es preferible a Z-Score** para la detección de outliers.

---
## 1.3 Método IQR (Rango Intercuartil)

El método del boxplot, formalizado por John Tukey (1977). Define outliers como valores fuera de las "cercas":

$$\text{IQR} = Q_3 - Q_1$$
$$\text{Límite Inferior} = Q_1 - 1.5 \times \text{IQR}$$
$$\text{Límite Superior} = Q_3 + 1.5 \times \text{IQR}$$

Cualquier valor fuera de estos límites es un outlier. El factor 1.5 es el estándar; con 3.0 se detectan solo outliers extremos.

In [ ]:
import numpy as np

def detectar_outliers_iqr(datos, multiplicador=1.5):
    """
    Detecta outliers usando método IQR
    """
    Q1 = np.percentile(datos, 25)
    Q3 = np.percentile(datos, 75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - multiplicador * IQR
    limite_superior = Q3 + multiplicador * IQR
    
    outliers_mask = (datos < limite_inferior) | (datos > limite_superior)
    
    return {
        'Q1': Q1,
        'Q3': Q3,
        'IQR': IQR,
        'limite_inferior': limite_inferior,
        'limite_superior': limite_superior,
        'outliers': datos[outliers_mask],
        'indices': np.where(outliers_mask)[0]
    }

# Usar función
siniestros = np.array([5, 8, 10, 12, 15, 18, 22, 25, 28, 100])
resultado = detectar_outliers_iqr(siniestros)

print(f"Q1: {resultado['Q1']:.2f}")
print(f"Q3: {resultado['Q3']:.2f}")
print(f"IQR: {resultado['IQR']:.2f}")
print(f"Límite Superior: {resultado['limite_superior']:.2f}")
print(f"Outliers detectados: {resultado['outliers']}")

### 💡 Observa:
- El IQR es 16.25K y el límite superior queda en ~50.125K
- El siniestro de 100K está claramente fuera del límite → **Outlier confirmado**
- Este método es **el mismo que usa el boxplot** para dibujar los "bigotes" y marcar los puntos fuera

**Ventaja del IQR:** No asume normalidad y es robusto a outliers (los cuartiles no se distorsionan por valores extremos).

---
## 1.4 Método de Percentiles

El asegurador define directamente el umbral de lo que considera "atípico" usando percentiles:

- **P95:** El 5% superior de los datos → outliers potenciales
- **P99:** El 1% superior → outliers extremos

**Ventaja en seguros:** Es la forma más directa de definir umbrales para reaseguro, reservas catastróficas y detección de fraude. El actuario decide el percentil según el apetito de riesgo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

siniestros = np.array([5, 8, 10, 12, 15, 18, 22, 25, 28, 100])

# Calcular percentiles
p25 = np.percentile(siniestros, 25)
p50 = np.percentile(siniestros, 50)
p75 = np.percentile(siniestros, 75)
p90 = np.percentile(siniestros, 90)
p95 = np.percentile(siniestros, 95)
p99 = np.percentile(siniestros, 99)

print(f"P25: {p25}")
print(f"P50 (Mediana): {p50}")
print(f"P75: {p75}")
print(f"P90: {p90}")
print(f"P95: {p95}")
print(f"P99: {p99}")

# Visualizar con histograma
plt.figure(figsize=(10, 6))
plt.hist(siniestros, bins=15, edgecolor='black', alpha=0.7)
plt.axvline(p95, color='red', linestyle='--', label=f'P95: {p95:.1f}K')
plt.axvline(p99, color='orange', linestyle='--', label=f'P99: {p99:.1f}K')
plt.xlabel('Monto de Siniestro (K)')
plt.ylabel('Frecuencia')
plt.legend()
plt.title('Distribución de Siniestros con Percentiles')
plt.show()

### 💡 Aplicación práctica en seguros:
- **P95** se usa frecuentemente como umbral para winsorización en modelos de pricing
- **P99** define el punto de retención en contratos de reaseguro ("los siniestros por encima de P99 van a reaseguro")
- La CNSF y Solvencia II usan VaR al 99.5% como medida de capital requerido

---
# 📊 TEMA 2: Visualización de Outliers

La visualización es **indispensable** para complementar los métodos estadísticos. Un número (Z-Score, IQR) no te dice si un outlier es un error de captura o un mega-siniestro legítimo. Las gráficas te dan el contexto para decidir.

Cubriremos 4 tipos de visualización:
- **Boxplot:** Visualización de la distribución y outliers por grupo
- **Scatterplot:** Relación entre dos variables, destaca puntos inusuales
- **Q-Q Plot:** ¿Los datos siguen una distribución normal? ¿Hay colas pesadas?
- **Densidad:** Distribución suave con outliers marcados

## 2.1 Diagrama de Caja (Boxplot)

El boxplot es la visualización más utilizada para identificar outliers. Muestra:
- **Caja:** Del Q1 al Q3 (rango intercuartil, 50% central de los datos)
- **Línea central:** Mediana (P50)
- **Bigotes:** Se extienden hasta Q1 − 1.5×IQR y Q3 + 1.5×IQR
- **Puntos fuera:** Outliers confirmados por el método IQR

Es especialmente útil para **comparar distribuciones entre grupos** (años, ramos, coberturas).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Datos de múltiples años
ano_2021 = np.random.normal(25, 5, 100)  # Media 25, SD 5
ano_2022 = np.random.normal(28, 6, 100)  # Media 28, SD 6
ano_2023 = np.random.normal(27, 7, 100)  # Media 27, SD 7

# Añadir algunos outliers deliberados
ano_2021 = np.append(ano_2021, [150, 160])  # 2 outliers
ano_2022 = np.append(ano_2022, [180])        # 1 outlier
ano_2023 = np.append(ano_2023, [200, 210, 215])  # 3 outliers

# Crear boxplot
fig, ax = plt.subplots(figsize=(10, 6))

bp = ax.boxplot([ano_2021, ano_2022, ano_2023],
                 labels=['2021', '2022', '2023'],
                 notch=True,           # Añade muescas
                 patch_artist=True)    # Permite colorear

# Colorear cajas
colors = ['lightblue', 'lightgreen', 'lightcoral']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_ylabel('Monto de Siniestro ($K)', fontsize=12)
ax.set_xlabel('Año', fontsize=12)
ax.set_title('Distribución de Siniestros por Año (Boxplot)', 
             fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Estadísticas
print("Año 2021:")
print(f"  Q1: {np.percentile(ano_2021, 25):.2f}")
print(f"  Mediana: {np.median(ano_2021):.2f}")
print(f"  Q3: {np.percentile(ano_2021, 75):.2f}")
print(f"  Outliers: {ano_2021[ano_2021 > 100]}")

### 💡 Cómo interpretar:
- **2023 tiene más outliers** y mayor variabilidad (caja más ancha) → posible deterioro del portafolio
- Las **muescas** muestran el intervalo de confianza de la mediana; si no se solapan entre años, las medianas son significativamente diferentes
- Los puntos rojos fuera de los bigotes son los outliers IQR

**Pregunta para el alumno:** ¿Los mega-siniestros de 2023 los eliminarías del modelo de pricing? ¿Por qué sí o no?

---
## 2.2 Gráfico de Dispersión (Scatterplot)

El scatterplot muestra la **relación entre dos variables** y permite detectar outliers que son inusuales en el contexto de esa relación. Un siniestro puede no ser outlier por monto, pero sí ser atípico para su antigüedad de póliza.

**Uso típico en seguros:** Monto de siniestro vs. antigüedad de póliza, edad del conductor, suma asegurada, etc.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Generar datos realistas
np.random.seed(42)
antiguedad = np.random.uniform(0, 10, 500)  # años
siniestro = 15 + 2*antiguedad + np.random.normal(0, 5, 500)  # relación lineal

# Añadir outliers deliberados
outlier_indices = np.random.choice(500, 15, replace=False)
siniestro[outlier_indices] = np.random.uniform(150, 200, 15)

# Crear gráfico
fig, ax = plt.subplots(figsize=(12, 7))

# Graficar puntos normales
normal_mask = np.ones(500, dtype=bool)
normal_mask[outlier_indices] = False

ax.scatter(antiguedad[normal_mask], siniestro[normal_mask], 
          alpha=0.6, label='Normal', s=50, color='blue')

# Graficar outliers
ax.scatter(antiguedad[outlier_indices], siniestro[outlier_indices],
          alpha=0.8, label='Outliers', s=100, color='red', marker='X')

ax.set_xlabel('Antigüedad de Póliza (años)', fontsize=12)
ax.set_ylabel('Monto de Siniestro ($K)', fontsize=12)
ax.set_title('Relación entre Antigüedad de Póliza y Monto de Siniestro',
            fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Estadísticas de outliers
print(f"Outliers detectados: {len(outlier_indices)}")
print(f"Montos de outliers: {siniestro[outlier_indices]}")

### 💡 Observa:
- Los puntos normales siguen una relación lineal (a mayor antigüedad, siniestro ligeramente mayor)
- Los outliers (puntos rojos ✕) están **completamente fuera del patrón** de la nube de puntos
- En la práctica: estos podrían ser fraudes, errores de captura o mega-siniestros legítimos

**Pregunta:** ¿Un siniestro de $180K en una póliza de 2 años de antigüedad es más sospechoso que uno de $180K en una póliza de 9 años? ¿Por qué?

---
## 2.3 Gráfico Q-Q (Quantile-Quantile)

El Q-Q Plot compara la **distribución real de los datos** vs. una **distribución teórica** (generalmente normal). Si los datos son normales, los puntos siguen la línea diagonal.

**Interpretación clave:**
- Puntos **sobre la línea** → distribución normal
- Puntos que **se curvan hacia arriba** en la cola derecha → **colas pesadas** (típico en seguros)
- Puntos que **se desvían** en ambas colas → outliers en ambas direcciones

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np

# Datos con distribución sesgada
np.random.seed(42)
siniestros_normales = np.random.normal(50, 15, 200)
siniestros_sesgados = np.concatenate([siniestros_normales, 
                                      np.random.uniform(100, 300, 20)])

# Crear figura con dos subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# QQ-plot para datos normales
stats.probplot(siniestros_normales, dist="norm", plot=ax1)
ax1.set_title('Q-Q Plot: Datos Aproximadamente Normales')
ax1.grid(True, alpha=0.3)

# QQ-plot para datos sesgados
stats.probplot(siniestros_sesgados, dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot: Datos con Colas Pesadas (Outliers)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Interpretación
print("Datos Normales:")
print(f"  Skewness: {stats.skew(siniestros_normales):.3f}")
print(f"  Kurtosis: {stats.kurtosis(siniestros_normales):.3f}")

print("\nDatos Sesgados:")
print(f"  Skewness: {stats.skew(siniestros_sesgados):.3f}")
print(f"  Kurtosis: {stats.kurtosis(siniestros_sesgados):.3f}")

### 💡 Observa la diferencia:
- **Izquierda (datos normales):** Los puntos siguen la línea diagonal casi perfectamente. Skewness ≈ 0.
- **Derecha (datos con outliers):** Los puntos se desvían fuertemente en la cola derecha, curvándose hacia arriba. Esto indica **colas pesadas** — exactamente lo que vemos en distribuciones de siniestros.

**En seguros:** Si el Q-Q Plot muestra colas pesadas, **no uses Z-Score** para detectar outliers (asume normalidad). Usa MAD o IQR que son robustos.

---
## 2.4 Gráfico de Densidad con Outliers

El gráfico de densidad (KDE — Kernel Density Estimation) muestra la **distribución suave** de los datos y permite ver dónde están los outliers en relación con la masa principal de la distribución.

**Ventaja sobre el histograma:** No depende del número de bins; muestra una curva suave.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

# Generar datos
np.random.seed(42)
datos = np.random.lognormal(3, 1, 500)  # Distribución sesgada

# Detectar outliers con IQR
Q1 = np.percentile(datos, 25)
Q3 = np.percentile(datos, 75)
IQR = Q3 - Q1
limite_sup = Q3 + 1.5 * IQR

outliers_mask = datos > limite_sup

# Crear figura
fig, ax = plt.subplots(figsize=(12, 7))

# Graficar densidad
density = gaussian_kde(datos)
xs = np.linspace(0, max(datos), 200)
ax.plot(xs, density(xs), 'b-', linewidth=2, label='Densidad')
ax.fill_between(xs, density(xs), alpha=0.3)

# Marcar outliers
outliers = datos[outliers_mask]
ax.scatter(outliers, density(outliers), color='red', s=100, 
          label=f'Outliers (n={len(outliers)})', zorder=5)

# Marcar umbral
ax.axvline(limite_sup, color='red', linestyle='--', 
          label=f'Umbral IQR: ${limite_sup:.0f}')

ax.set_xlabel('Monto de Siniestro ($)', fontsize=12)
ax.set_ylabel('Densidad de Probabilidad', fontsize=12)
ax.set_title('Densidad de Siniestros con Outliers Identificados',
            fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 💡 Observa:
- La distribución es claramente **log-normal** (sesgada a la derecha)
- Los puntos rojos a la derecha del umbral IQR son los outliers
- La densidad decae rápidamente después del pico, pero los outliers están en la cola

**En seguros:** Esta forma de distribución es típica de siniestros de propiedad, auto y gastos médicos. Los outliers de la cola derecha representan los siniestros catastróficos que definen las capas de reaseguro.

---
### 🎯 Resumen de Aprendizajes

| Concepto | Aplicación | Riesgo de no aplicar correctamente |
|----------|------------|-----------------------------------|
| Z-Score | Detección rápida y sencilla | Falsos positivos en datos sesgados |
| Z-Modificado (MAD) | Robusto, no asume normalidad | Más complejo de explicar |
| IQR | Método estándar y robusto | Puede omitir sutilezas |
| Boxplots | Visualización clara e intuitiva | Interpretación errónea si no se conoce |
| 
| Percentiles | Define umbrales de negocio | Subjetividad en la elección del percentil |

**⚠️ Recuerda:** Un outlier no siempre es un error. En seguros, puede ser la pérdida más importante que debas modelar.